# Full visual audit: is the `palpebral` mask correctly on the tissue for EVERY aligned patient?

**Why this notebook exists (and why a 10-patient sample isn't enough):** `verify_alignment_sanity_check.ipynb` already visually checked a 10-patient sample (always including `India_071`, the patient that exposed the earlier `matchTemplate` failure) and it passed. But a sample can miss a per-patient outlier -- the exact failure mode this project has hit before was a mask that looked completely plausible in aggregate stats while being spatially wrong for one specific patient. This notebook is meant to be **run and looked at by you, personally, for all 201 successfully-aligned patients** (`data/processed/aligned_raw/`) -- not just something I summarize.

**Format, per patient:** two panels side by side --
1. The full raw eye photo, on its own.
2. The same photo with the mask overlaid (semi-transparent red).

The red patch in panel 2 should sit on the lower everted eyelid's conjunctiva (a curved strip just below the eye), matching real tissue you can see in panel 1. Watch for the red patch floating on the sclera (white of the eye), on skin, on empty space, or forming an unnaturally sharp rectangle instead of a soft curved strip.

Run every cell, then scroll through all 201 pairs. The automated blank/near-blank scan at the end is a useful secondary check but can't catch "non-blank but in the wrong place" -- only your own look-through can.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# Assumes the notebook is run from within notebooks/ (Jupyter's default cwd
# when opened from that folder). If running elsewhere, set PROJECT_ROOT directly.
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

ALIGNED_ROOT = PROCESSED_DIR / "aligned_raw"
ALIGNED_IMAGES_DIR = ALIGNED_ROOT / "images"
ALIGNED_MASKS_DIR = ALIGNED_ROOT / "masks"
ALIGNMENT_LOG_CSV = ALIGNED_ROOT / "alignment_log.csv"

print("PROJECT_ROOT:        ", PROJECT_ROOT)
print("ALIGNED_IMAGES_DIR exists:", ALIGNED_IMAGES_DIR.exists())
print("ALIGNED_MASKS_DIR exists: ", ALIGNED_MASKS_DIR.exists())
print("ALIGNMENT_LOG_CSV exists: ", ALIGNMENT_LOG_CSV.exists())

alignment_log = pd.read_csv(ALIGNMENT_LOG_CSV).set_index("patient_id")
ok_patient_ids = sorted(alignment_log[alignment_log["status"] == "ok"].index.tolist())
excluded = alignment_log[alignment_log["status"] != "ok"]

print(f"\n{len(ok_patient_ids)} successfully aligned patients to audit, one pair of panels each below")
print(f"{len(excluded)} excluded (no aligned_raw output exists for them, not part of this audit):")
print(excluded["status"])

## Every patient, full image then image+mask overlay

This produces 201 figures, one per patient, in order. Scroll through all of them.

In [ ]:
def make_overlay(image: np.ndarray, mask: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    mask_bin = (mask > 127).astype(np.float32)
    red = np.zeros_like(image, dtype=np.float32)
    red[..., 0] = 255
    blended = image.astype(np.float32) * (1 - alpha * mask_bin[..., None]) + red * (alpha * mask_bin[..., None])
    return blended.astype(np.uint8)


for pid in ok_patient_ids:
    image = np.array(Image.open(ALIGNED_IMAGES_DIR / f"{pid}.jpg").convert("RGB"))
    mask = np.array(Image.open(ALIGNED_MASKS_DIR / f"{pid}.png").convert("L"))
    overlay = make_overlay(image, mask)

    log_row = alignment_log.loc[pid]
    n_positive = int((mask > 0).sum())

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(
        f"{pid}   ({log_row['method']}, {int(log_row['n_inliers'])} inliers, "
        f"mask_source={log_row['mask_source']}, {n_positive} positive px)",
        fontsize=12,
    )
    axes[0].imshow(image)
    axes[0].set_title("1. Full raw eye photo")
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title("2. Same photo + mask overlay")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

## Quantitative full-scan (blank/near-blank check)

Automated complement to the visual check above -- catches a fully/nearly empty mask explicitly, which is easy to eyeball but easy to also just confirm here.

In [ ]:
records = []
for pid in ok_patient_ids:
    mask = np.array(Image.open(ALIGNED_MASKS_DIR / f"{pid}.png").convert("L"))
    n_positive = int((mask > 0).sum())
    records.append({"patient_id": pid, "positive_px": n_positive, "positive_frac": n_positive / mask.size})

scan_df = pd.DataFrame(records)
print(scan_df["positive_px"].describe())

blank_masks = scan_df[scan_df["positive_px"] == 0]
print(f"\nFully blank masks (0 positive pixels): {len(blank_masks)} / {len(scan_df)}")
if len(blank_masks):
    print(blank_masks)

low_signal = scan_df[scan_df["positive_px"] < 10]
print(f"\nNear-blank masks (<10 positive pixels): {len(low_signal)} / {len(scan_df)}")
if len(low_signal):
    print(low_signal)

plt.figure(figsize=(8, 4))
plt.hist(scan_df["positive_frac"] * 100, bins=50)
plt.xlabel("Foreground percentage per mask")
plt.ylabel("Number of patients")
plt.title("Distribution of foreground fraction across all 201 aligned_raw masks")
plt.show()

## Verdict

Fill in after actually scrolling through every pair above:
- [ ] All 201 pairs show the red overlay sitting on the lower eyelid conjunctiva, not the sclera/skin/empty space.
- [ ] No blank or near-blank masks (should already read 0/201 above if the alignment pipeline hasn't changed).
- [ ] Any exceptions found, noted here with patient_id and what was wrong: